# mistral_common smoke test

Example 2 needs the SentencePiece tokenizer, Example 3 the llguidance converter and Example 4 the audio
decoders, none of which a plain `pip install mistral_common` installs. Install the package with those extras
to run this notebook: `pip install "mistral_common[sentencepiece,guidance,audio]"`. Example 5 also encodes the
image when OpenCV (`cv2`) is installed, and checks the error message when it is not.

## Example 1 — the installed version; the bundled Tekken tokenizer encodes a chat request that carries a tool

In [ ]:
import importlib.metadata
import mistral_common
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import Function, Tool
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

assert mistral_common.__version__ == importlib.metadata.version("mistral_common"), mistral_common.__version__
tool = Tool(function=Function(name="get_weather", description="Weather for a city",
                              parameters={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}))
request = ChatCompletionRequest(tools=[tool], messages=[UserMessage(content="What is the weather in Paris?")])
tokenizer = MistralTokenizer.v3(is_tekken=True)          # loaded from the wheel's own data folder: no download
assert tokenizer.instruct_tokenizer.tokenizer.n_words == 131072
encoded = tokenizer.encode_chat_completion(request)
assert encoded.text.startswith("<s>[AVAILABLE_TOOLS]") and "[INST]What is the weather in Paris?[/INST]" in encoded.text, encoded.text
decoded = tokenizer.decode(encoded.tokens)
assert '"name": "get_weather"' in decoded and "What is the weather in Paris?" in decoded, decoded
print("mistral_common", mistral_common.__version__ + ": Tekken (131,072 tokens) encoded the request into", len(encoded.tokens), "tokens")

## Example 2 — the SentencePiece tokenizer encodes the same request with its own vocabulary

In [ ]:
from mistral_common.protocol.instruct.messages import UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import Function, Tool
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

tool = Tool(function=Function(name="get_weather", description="Weather for a city",
                              parameters={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}))
request = ChatCompletionRequest(tools=[tool], messages=[UserMessage(content="What is the weather in Paris?")])
sentencepiece = MistralTokenizer.v3()
tekken = MistralTokenizer.v3(is_tekken=True)
assert type(sentencepiece.instruct_tokenizer.tokenizer).__name__ == "SentencePieceTokenizer"
assert sentencepiece.instruct_tokenizer.tokenizer.n_words == 32768
by_sentencepiece = sentencepiece.encode_chat_completion(request)
by_tekken = tekken.encode_chat_completion(request)
assert by_sentencepiece.tokens != by_tekken.tokens
for decoded in (sentencepiece.decode(by_sentencepiece.tokens), tekken.decode(by_tekken.tokens)):
    assert '"name": "get_weather"' in decoded and decoded.endswith("What is the weather in Paris?"), decoded
print("SentencePiece (32,768 tokens):", len(by_sentencepiece.tokens), "tokens; Tekken:", len(by_tekken.tokens), "; both decode back to the request")

## Example 3 — a malformed conversation is refused; the Tekken tokenizer converts for grammar-constrained output

In [ ]:
from mistral_common.exceptions import InvalidMessageStructureException
from mistral_common.guidance.tokenizer import from_mistral_tokenizer
from mistral_common.protocol.instruct.messages import AssistantMessage, UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

tokenizer = MistralTokenizer.v3(is_tekken=True)
try:
    tokenizer.encode_chat_completion(ChatCompletionRequest(messages=[AssistantMessage(content="I speak first")]))
    raise AssertionError("a conversation that starts with the assistant was accepted")
except InvalidMessageStructureException as error:
    print("refused:", error)
turns = [UserMessage(content="Hi"), AssistantMessage(content="Hello!"), UserMessage(content="Bye")]
assert "[INST]Hi[/INST]Hello!</s>[INST]Bye[/INST]" in tokenizer.encode_chat_completion(ChatCompletionRequest(messages=turns)).text
for_grammar = from_mistral_tokenizer(tokenizer)          # the `guidance` extra: an llguidance tokenizer
assert type(for_grammar).__name__ == "LLTokenizer" and for_grammar.vocab_size == 131072, (type(for_grammar), for_grammar.vocab_size)
print("three turns encoded; llguidance tokenizer built with", for_grammar.vocab_size, "tokens")

## Example 4 — audio input: a WAV is decoded (soundfile) and resampled to 16 kHz (soxr)

In [ ]:
import base64
import io
import numpy as np
import soundfile
from mistral_common.tokens.tokenizers.audio import Audio

rate, seconds = 44100, 2
tone = (0.3 * np.sin(2 * np.pi * 440.0 * np.arange(rate * seconds) / rate)).astype(np.float32)
wav = io.BytesIO()
soundfile.write(wav, tone, rate, format="WAV")
audio = Audio.from_base64(base64.b64encode(wav.getvalue()).decode())
assert (audio.sampling_rate, audio.audio_array.shape, audio.format) == (rate, (rate * seconds,), "wav")
audio.resample(16000)
assert (audio.sampling_rate, audio.audio_array.shape) == (16000, (16000 * seconds,)), (audio.sampling_rate, audio.audio_array.shape)
spectrum = np.abs(np.fft.rfft(audio.audio_array))
assert abs(float(np.argmax(spectrum)) / seconds - 440.0) < 1.0
print("2 s of 44.1 kHz audio decoded and resampled to", audio.audio_array.shape[0], "samples at 16 kHz; the tone is still 440 Hz")

## Example 5 — image input: the request is built; encoding the image needs OpenCV (cv2)

In [ ]:
import importlib.util
from PIL import Image
from mistral_common.protocol.instruct.messages import ImageChunk, TextChunk, UserMessage
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

picture = Image.new("RGB", (64, 48), (255, 0, 0))
request = ChatCompletionRequest(messages=[UserMessage(content=[TextChunk(text="Describe it"), ImageChunk(image=picture)])])
assert request.messages[0].content[1].image.size == (64, 48)
tokenizer = MistralTokenizer.v3(is_tekken=True, is_mm=True)
if importlib.util.find_spec("cv2") is None:
    try:
        tokenizer.encode_chat_completion(request)
        raise AssertionError("an image was encoded without OpenCV")
    except ImportError as error:
        assert "opencv" in str(error), error
        print("SKIP: `cv2` (OpenCV) is not installed -> image encoding refused:", error)
else:
    encoded = tokenizer.encode_chat_completion(request)
    assert len(encoded.images) == 1 and encoded.images[0].shape[0] == 3, [image.shape for image in encoded.images]
    print("image encoded into", len(encoded.tokens), "tokens and one array of shape", encoded.images[0].shape)